# Chapter 8 — Support-Vector Machines

This notebook corresponds to Week 8 of *Applied Machine Learning* and uses the Flight Price Prediction dataset. Place the book's `Data-Week8.csv` file in `data/raw/` at the repository root.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.metrics import accuracy_score

data_path = Path('../../data/raw/Data-Week8.csv')
data = pd.read_csv(data_path)
data.head()

## Prepare the two-feature example

The assignment form below produces the same destination codes as the book while avoiding deprecated pandas chained assignment. The book uses duration and price to classify destination city.


In [ ]:
data['destination_city'] = data['destination_city'].replace(
    ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai'],
    [0, 1, 2, 3, 4, 5],
)

X = data[['duration', 'price']]
Y = data['destination_city']
x = X.iloc[:1500, :]
y = Y.iloc[:1500]

## Compare four SVM kernels

> Methodology note: the models are fitted and scored on the same 1,500 observations, as in the book. These figures are training accuracies rather than held-out estimates.


In [ ]:
svc = svm.SVC(kernel='linear', C=1).fit(x, y)
svc_pred = svc.predict(x)
print('svc accuracy', accuracy_score(y, svc_pred) * 100)

linear_svc = svm.LinearSVC(C=1).fit(x, y)
linear_svc_pred = linear_svc.predict(x)
print('linear svc accuracy', accuracy_score(y, linear_svc_pred) * 100)

rbf_svc = svm.SVC(kernel='rbf', gamma=0.7, C=1).fit(x, y)
rbf_svc_pred = rbf_svc.predict(x)
print('rbf accuracy', accuracy_score(y, rbf_svc_pred) * 100)

poly_svc = svm.SVC(kernel='poly', degree=3, C=1).fit(x, y)
poly_svc_pred = poly_svc.predict(x)
print('poly accuracy', accuracy_score(y, poly_svc_pred) * 100)

## Decision regions


In [ ]:
h = 10
x_min, x_max = x.iloc[:, 0].min() - 1, x.iloc[:, 0].max() + 1
y_min, y_max = x.iloc[:, 1].min() - 1, x.iloc[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h), np.arange(y_min, y_max, h)
)

titles = ['linear kernel', 'linear svc', 'rbf', 'polynomial']
for i, classifier in enumerate((svc, linear_svc, rbf_svc, poly_svc)):
    plt.subplot(2, 2, i + 1)
    plt.subplots_adjust(wspace=0.4, hspace=0.4)
    Z = classifier.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.8)
    plt.scatter(x.iloc[:, 0], x.iloc[:, 1], c=y, cmap=plt.cm.coolwarm)
    plt.ylabel('price')
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.xticks(())
    plt.yticks(())
    plt.title(titles[i])
plt.show()

Book results: linear-kernel SVC `72.8667%`, LinearSVC `58.4667%`, RBF `100%`, and polynomial `71.4%`. The book identifies the RBF model as the most effective on these training observations.
